# 问题 3：流派比较

比较流派内/流派间的相似度与影响力；回答「什么区分一个流派」「流派如何随时间变化」「流派之间是否相关」。

**内容：**
1. 流派画像热图
2. 区分流派的特征（异常值）
3. 流派间影响矩阵（原始 + 归一化密度）
4. 流派相似度矩阵
5. 相似度 vs 影响力的一致性
6. 流派特征随时间轨迹

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 100

artists = pd.read_csv('output/artists.csv', dtype=str)
edges = pd.read_csv('output/edges.csv', dtype=str)

FEATURES = ['danceability', 'energy', 'valence', 'tempo', 'loudness',
            'acousticness', 'instrumentalness', 'liveness', 'speechiness', 'duration_ms']
for f in FEATURES:
    artists[f] = pd.to_numeric(artists[f], errors='coerce')

labeled = artists[artists['genre'] != 'Unknown'].copy()
Z = (labeled[FEATURES] - labeled[FEATURES].mean()) / labeled[FEATURES].std()
labeled = pd.concat([labeled, Z.add_suffix('_z')], axis=1)
Z_FEATURES = [f + '_z' for f in FEATURES]

print('有流派标签的艺术家:', len(labeled))
print('流派数:', labeled['genre'].nunique())

## 1. 流派画像热图

In [ ]:
top_genres = labeled['genre'].value_counts().head(12).index.tolist()
sub = labeled[labeled['genre'].isin(top_genres)]

profile = sub.groupby('genre')[Z_FEATURES].mean().loc[top_genres]

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(profile.values, cmap='RdBu_r', aspect='auto')
ax.set_xticks(range(len(FEATURES)))
ax.set_xticklabels(FEATURES, rotation=45, ha='right')
ax.set_yticks(range(len(top_genres)))
ax.set_yticklabels(top_genres)
plt.colorbar(im, ax=ax, label='standardized mean')
ax.set_title('Genre profiles (standardized feature means)')
plt.tight_layout(); plt.show()

## 2. 区分流派的特征（异常值）

某些流派在特定特征上极端，可用异常值识别。以 `acousticness` 和 `instrumentalness` 为例。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, feat in zip(axes, ['acousticness', 'instrumentalness']):
    data = [labeled[labeled['genre'] == g][feat].dropna().values for g in top_genres]
    ax.boxplot(data, tick_labels=top_genres, vert=False, showfliers=False)
    ax.set_xlabel(feat)
    ax.set_title(f'{feat} distribution by genre')
plt.tight_layout(); plt.show()

## 3. 流派间影响矩阵

把影响边按「影响者流派 → 追随者流派」汇总，得到流派间影响流；再做归一化密度，避免规模误导。

In [ ]:
flow = (edges.groupby(['influencer_main_genre', 'follower_main_genre'])
        .size().unstack(fill_value=0))
flow = flow.loc[top_genres, top_genres]  # 只看 top 流派

# 归一化：除以「可能连接数」= 影响者流派人数 × 追随者流派人数
count = labeled['genre'].value_counts()
possible = np.outer(count.loc[top_genres].values, count.loc[top_genres].values)
flow_density = flow / possible

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(np.log1p(flow.values), cmap='Blues', aspect='auto')
axes[0].set_title('Raw influence flow (log1p)')
axes[1].imshow(flow_density.values, cmap='Blues', aspect='auto')
axes[1].set_title('Normalized density (per possible link)')
for ax in axes:
    ax.set_xticks(range(len(top_genres))); ax.set_xticklabels(top_genres, rotation=45, ha='right')
    ax.set_yticks(range(len(top_genres))); ax.set_yticklabels(top_genres)
    ax.set_xlabel('follower genre'); ax.set_ylabel('influencer genre')
plt.tight_layout(); plt.show()

## 4. 流派相似度矩阵

In [ ]:
# 流派中心向量（标准化特征均值），两两算相似度 S=1/(1+d)
centers = profile.values
sim_mat = np.zeros((len(top_genres), len(top_genres)))
for i in range(len(top_genres)):
    for j in range(len(top_genres)):
        d = np.linalg.norm(centers[i] - centers[j])
        sim_mat[i, j] = 1.0 / (1.0 + d)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim_mat, cmap='RdBu_r', vmin=0.4, vmax=1.0)
ax.set_xticks(range(len(top_genres))); ax.set_xticklabels(top_genres, rotation=45, ha='right')
ax.set_yticks(range(len(top_genres))); ax.set_yticklabels(top_genres)
plt.colorbar(im, ax=ax, label='similarity')
ax.set_title('Genre similarity matrix (S = 1/(1+d))')
plt.tight_layout(); plt.show()

## 5. 相似度 vs 影响力的一致性

把相似度矩阵和影响密度矩阵的上三角拉平，看「流派之间越相似是否影响也越强」。

In [ ]:
iu = np.triu_indices(len(top_genres), k=1)
sim_flat = sim_mat[iu]
flow_flat = flow_density.values[iu]

r = np.corrcoef(sim_flat, flow_flat)[0, 1]
print(f'相似度 与 影响密度 的相关系数 = {r:.3f}')

plt.figure(figsize=(7, 6))
plt.scatter(sim_flat, flow_flat, alpha=0.7, color='#4C72B0')
plt.xlabel('Genre similarity')
plt.ylabel('Influence density')
plt.title(f'Similarity vs influence (r={r:.3f})')
plt.tight_layout(); plt.show()

## 6. 流派特征随时间轨迹

用歌曲级数据按流派+年代聚合，看关键特征如何变化。

In [ ]:
songs = pd.read_csv('output/songs.csv', dtype=str)
song_artist = pd.read_csv('output/song_artist.csv', dtype=str)
genre_map = artists.set_index('artist_id')['genre']

sa = song_artist.copy()
sa['genre'] = sa['artist_id'].map(genre_map)

songs['year'] = pd.to_numeric(songs['year'], errors='coerce')
song_feat = songs[FEATURES + ['year']].copy()
song_feat['song_index'] = song_feat.index
for f in FEATURES:
    song_feat[f] = pd.to_numeric(song_feat[f], errors='coerce')

sa['song_index'] = sa['song_index'].astype(int)
sa = sa.merge(song_feat, on='song_index', how='left')
sa['decade'] = (sa['year'] // 10 * 10).astype('Int64')
sa = sa.dropna(subset=['decade'])

focus_genres = ['Pop/Rock', 'Jazz', 'Blues', 'Country', 'R&B']
sa_focus = sa[sa['genre'].isin(focus_genres)]
trend = sa_focus.groupby(['genre', 'decade'])['energy'].mean().unstack()

plt.figure(figsize=(10, 5))
for g in focus_genres:
    if g in trend.index:
        plt.plot(trend.columns, trend.loc[g], marker='o', label=g)
plt.xlabel('Decade')
plt.ylabel('Mean energy')
plt.title('Energy by genre over time')
plt.legend()
plt.tight_layout(); plt.show()

## 7. 结论

- 画像热图显示各流派在特征空间有清晰差异（如 Jazz/Classical 偏原声、Pop/Rock 偏律动）。
- 影响矩阵 vs 相似度矩阵的相关性回答了「流派之间是「像」还是「互相影响」」——相关高说明相似与影响一致。
- 时间轨迹显示流派特征随年代漂移，可进一步与问题 5 的革命检测衔接。